In [ ]:
import pytesseract

# UPDATE THIS PATH if yours is different
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

In [ ]:
# ---
# Michigan Driver's License Fake-ID Prototype (OCR + Rules)
# ---

!apt-get install -y tesseract-ocr
!pip install -q pytesseract pillow opencv-python numpy python-dateutil
!pip install pdf417decoder

import cv2
import pytesseract
import numpy as np
from PIL import Image
import re
from dateutil import parser
from google.colab import files

from pdf417decoder import PDF417Decoder
from PIL import Image as PIL  # You already import PIL.Image as Image, this adds alias

pytesseract.pytesseract.tesseract_cmd = "/usr/bin/tesseract"

print("Setup complete.")

In [ ]:
print("Upload a scanned or sample Michigan driver's license FRONT image...")
from tkinter import Tk, filedialog
import cv2

# Use a hidden Tkinter window to open file explorer
Tk().withdraw()

print("Select your FRONT ID image...")
front_path = filedialog.askopenfilename(
    title="Select Front of ID",
    filetypes=[("Image Files", "*.png *.jpg *.jpeg *.bmp")]
)

img = cv2.imread(front_path)
rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
H, W, _ = rgb.shape

print(f"Loaded front ID: {front_path}")
Image.fromarray(rgb)

# Normalize size so our ROI fractions make sense
h, w = img.shape[:2]
TARGET_W = 1200
scale = TARGET_W / w
img = cv2.resize(img, (TARGET_W, int(h * scale)))

rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
H, W, _ = rgb.shape

print(f"Image loaded with shape: {rgb.shape}")
Image.fromarray(rgb)

In [ ]:
# y1, x1, y2, x2 as FRACTIONS of height/width
# These assume: photo on the left, text on the right.
ROIS = {
    "name":       (0.38, 0.30, 0.44, 0.63),  # row with LAST FIRST
    "dob_line":   (0.31, 0.30, 0.38, 0.55),  # line with DOB + eyes, etc.
    "dates_row":  (0.25, 0.61, 0.38, 0.84),  # ISS / EXP row (top-right)
    "license_no": (0.25, 0.30, 0.31, 0.60),  # DL / ID number row (upper mid)
    "header": (0.03, 0.02, 0.14, 0.41)
}

def crop_roi(img, roi_frac):
    y1f, x1f, y2f, x2f = roi_frac
    h, w = img.shape[:2]
    y1, x1 = int(y1f * h), int(x1f * w)
    y2, x2 = int(y2f * h), int(x2f * w)
    return img[y1:y2, x1:x2]

# Draw debug boxes so you can see if ROIs are in the right place
debug = img.copy()
for label, frac in ROIS.items():
    y1f, x1f, y2f, x2f = frac
    y1, x1 = int(y1f * H), int(x1f * W)
    y2, x2 = int(y2f * H), int(x2f * W)
    cv2.rectangle(debug, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.putText(debug, label, (x1, max(y1-5, 15)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

Image.fromarray(cv2.cvtColor(debug, cv2.COLOR_BGR2RGB))


In [ ]:
custom_config_line  = r"--oem 3 --psm 7"  # treat as single line
custom_config_block = r"--oem 3 --psm 6"  # small text block

def ocr_roi(label, mode="line"):
    roi = crop_roi(rgb, ROIS[label])
    gray = cv2.cvtColor(roi, cv2.COLOR_RGB2GRAY)
    gray = cv2.threshold(
        gray, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )[1]
    cfg = custom_config_line if mode == "line" else custom_config_block
    text = pytesseract.image_to_string(gray, config=cfg)
    print(f"\n=== OCR for {label} ===")
    print(text)
    return text

text_name      = ocr_roi("name", mode="line")
text_dob_line  = ocr_roi("dob_line", mode="block")
text_dates_row = ocr_roi("dates_row", mode="block")
text_license   = ocr_roi("license_no", mode="block")
text_header = ocr_roi("header", mode="line")


In [ ]:
def find_date(text):
    # MM-DD-YYYY or MM/DD/YYYY
    m = re.search(r"(\d{2}[-/]\d{2}[-/]\d{4})", text)
    return m.group(1) if m else None

def find_license(text):
    # MI DL often like: S 000 123 456 789
    cleaned = text.replace(" ", "")
    m = re.search(r"\b([A-Z]\d{12})\b", cleaned)
    return m.group(1) if m else None

def clean_name(text):
    # Very simple: 2-3 ALLCAPS words (LAST FIRST MIDDLE)
    parts = re.findall(r"[A-Z]{2,}", text)
    if len(parts) >= 2:
        return " ".join(parts[:3])
    return None
def find_iss(text):
    m = re.search(r"ISS[:\s]*?(\d{2}[-/]\d{2}[-/]\d{4})", text, re.IGNORECASE)
    return m.group(1) if m else None

fields = {}
fields["name"]           = clean_name(text_name)
fields["dob"]            = find_date(text_dob_line)
fields["exp"]            = find_date(text_dates_row)
fields["issue"]          = find_iss(text_dates_row)
fields["license_number"] = find_license(text_dob_line + " " + text_license)

print("\n===== PARSED FIELDS (ROI-based) =====")
for k, v in fields.items():
    print(f"{k}: {v}")


In [ ]:
def valid_date(date_str):
    if not date_str:
        return None
    try:
        return parser.parse(date_str, dayfirst=False, yearfirst=False)
    except Exception:
        return None

results = []
score = 100  # start at 100, subtract for issues

# Rule 1: License number format
ln = fields["license_number"]
if ln:
    if not re.match(r"^[A-Z]\d{12}$", ln):
        score -= 20
        results.append("❌ Invalid license number format (should be 1 letter + 12 digits).")
else:
    score -= 25
    results.append("❌ License number missing or unreadable.")

# Rule 2: DOB valid, and age >= 21 (bar mode)
dob_str = fields["dob"]
dob_dt = valid_date(dob_str)
if not dob_dt:
    score -= 20
    results.append("❌ DOB unreadable or invalid.")
else:
    from datetime import datetime
    today = datetime.today()
    age = today.year - dob_dt.year - ((today.month, today.day) < (dob_dt.month, dob_dt.day))
    if age < 21:
        score -= 30
        results.append(f"❌ Person appears under 21 (age ≈ {age}).")

# Rule 3: Expiration date valid and not crazy
exp_str = fields["exp"]
exp_dt = valid_date(exp_str)
if not exp_dt:
    score -= 10
    results.append("❌ EXP date unreadable or missing.")
else:
    if exp_dt.year > 2045:
        score -= 10
        results.append("⚠️ EXP date is very far in the future.")
    if dob_dt and exp_dt < dob_dt:
        score -= 20
        results.append("❌ EXP date earlier than DOB (nonsense).")

# Rule 4: Michigan template word
all_text = (
    text_header +
    text_name +
    text_dob_line +
    text_dates_row +
    text_license
).upper()

if "MICHIGAN" not in all_text:
    score -= 15
    results.append("❌ 'MICHIGAN' not detected clearly.")

# Rule 5: Very rough face-region edge check (just a heuristic)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
h2, w2 = gray.shape
face_region = gray[int(0.20*h2):int(0.75*h2), int(0.05*w2):int(0.35*w2)]
edges = cv2.Canny(face_region, 50, 150)
if np.sum(edges) < 4000:
    score -= 10
    results.append("⚠️ Face/photo region looks unusually blank.")

# Clamp score
score = max(0, min(100, score))

print("\n===== VALIDATION RESULTS =====")
if not results:
    print("✓ No major issues detected.")
else:
    for r in results:
        print(r)

print(f"\n===== FAKE ID SCORE =====")
print(f"Confidence this ID is REAL: {score}/100")
print(f"Confidence this ID is FAKE: {100-score}/100")


In [ ]:
print("Upload a BACK image of the Michigan driver's license (with the PDF417 barcode)...")
from tkinter import Tk, filedialog
from PIL import Image as PIL_Image

# Hide Tkinter root window
Tk().withdraw()

print("Select your BACK ID image (PDF417 barcode side)...")

back_path = filedialog.askopenfilename(
    title="Select Back of ID",
    filetypes=[("Image Files", "*.png *.jpg *.jpeg *.bmp")]
)

back_img = PIL_Image.open(back_path)
print(f"Loaded back ID: {back_path}")
back_img

# Decode PDF417
decoder = PDF417Decoder(back_image)
num_barcodes = decoder.decode()

if num_barcodes > 0:
    barcode_raw = decoder.barcode_data_index_to_string(0)
    print("===== RAW PDF417 DATA =====")
    print(barcode_raw)
else:
    barcode_raw = ""
    print("❌ No PDF417 barcode detected on back image.")

In [ ]:
def parse_aamva(barcode_text: str):
    """
    Parse some common AAMVA fields from a US driver license PDF417.
    This is a simplified parser for demo / prototype use.
    """
    data = {}

    def get(code):
        # Find the code followed by the rest of that line
        m = re.search(rf"{code}([^\n\r]+)", barcode_text)
        return m.group(1).strip() if m else None

    # Common AAMVA codes:
    # DCS = Last name
    # DAC = First name
    # DCT = First + middle (sometimes)
    # DAQ = License number
    # DBA = Expiration date (YYYYMMDD or MMDDYYYY)
    # DBB = Date of birth (YYYYMMDD or MMDDYYYY)
    # DBD = Issue date
    # DAG = Street
    # DAI = City
    # DAJ = State
    # DAK = Zip

    last_name  = get("DCS")
    first_name = get("DAC") or get("DCT")
    license_no = get("DAQ")
    dob_raw    = get("DBB")
    exp_raw    = get("DBA")
    iss_raw    = get("DBD")
    street     = get("DAG")
    city       = get("DAI")
    state      = get("DAJ")
    zipcode    = get("DAK")

    # Helper to normalize AAMVA date formats
    def norm_date(d):
        if not d:
            return None
        d = d.strip()
        # Most modern AAMVA: YYYYMMDD
        if re.match(r"^\d{8}$", d):
            yyyy = int(d[0:4])
            mm   = int(d[4:6])
            dd   = int(d[6:8])
            return f"{mm:02d}/{dd:02d}/{yyyy}"
        # Legacy: MMDDCCYY or MMDDYYYY
        if re.match(r"^\d{8}$", d):
            mm   = int(d[0:2])
            dd   = int(d[2:4])
            yyyy = int(d[4:8])
            return f"{mm:02d}/{dd:02d}/{yyyy}"
        # Fallback: try parser
        dt = valid_date(d)
        return dt.strftime("%m/%d/%Y") if dt else None

    dob = norm_date(dob_raw)
    exp = norm_date(exp_raw)
    iss = norm_date(iss_raw)

    # Build barcode_fields
    data["first_name"] = first_name
    data["last_name"] = last_name
    data["full_name"] = f"{last_name} {first_name}".strip() if (last_name or first_name) else None
    data["license_number"] = license_no
    data["dob"] = dob
    data["exp"] = exp
    data["issue"] = iss
    data["street"] = street
    data["city"] = city
    data["state"] = state
    data["zipcode"] = zipcode

    return data

if barcode_raw:
    barcode_fields = parse_aamva(barcode_raw)
    print("\n===== PARSED BARCODE FIELDS =====")
    for k, v in barcode_fields.items():
        print(f"{k}: {v}")
else:
    barcode_fields = {}


In [ ]:
print("\n===== FRONT vs BARCODE CONSISTENCY CHECK =====")

if not barcode_fields:
    print("No barcode data available to compare.")
else:
    # We'll subtract points if critical mismatches are found
    # and add messages to `results`.
    # This assumes `results` and `score` exist from your previous validation cell.
    # If they don't, create them:
    if "results" not in globals():
        results = []
    if "score" not in globals():
        score = 100

    # Compare license number
    front_ln = fields.get("license_number")
    back_ln  = barcode_fields.get("license_number")
    if front_ln and back_ln:
        if front_ln.replace(" ", "") != back_ln.replace(" ", ""):
            score -= 25
            results.append(f"❌ License number mismatch: front='{front_ln}' vs barcode='{back_ln}'")
        else:
            results.append("✅ License number matches barcode.")

    # Compare DOB
    front_dob = fields.get("dob")
    back_dob  = barcode_fields.get("dob")
    if front_dob and back_dob:
        if valid_date(front_dob) and valid_date(back_dob):
            if valid_date(front_dob).date() != valid_date(back_dob).date():
                score -= 20
                results.append(f"❌ DOB mismatch: front='{front_dob}' vs barcode='{back_dob}'")
            else:
                results.append("✅ DOB matches barcode.")
    elif front_dob or back_dob:
        score -= 10
        results.append("⚠️ DOB only found on one side (front or barcode).")

    # Compare EXP
    front_exp = fields.get("exp")
    back_exp  = barcode_fields.get("exp")
    if front_exp and back_exp:
        if valid_date(front_exp) and valid_date(back_exp):
            if valid_date(front_exp).date() != valid_date(back_exp).date():
                score -= 15
                results.append(f"❌ EXP mismatch: front='{front_exp}' vs barcode='{back_exp}'")
            else:
                results.append("✅ EXP matches barcode.")
    elif front_exp or back_exp:
        score -= 5
        results.append("⚠️ EXP only found on one side (front or barcode).")

    # Rough name comparison (last name only)
    front_name = fields.get("name") or ""
    front_last = front_name.split()[0] if front_name else None
    back_last  = barcode_fields.get("last_name")
    if front_last and back_last:
        if front_last.upper() != back_last.upper():
            score -= 10
            results.append(f"⚠️ Last name mismatch: front='{front_last}' vs barcode='{back_last}'")
        else:
            results.append("✅ Last name matches barcode (rough check).")

    # Clamp score
    score = max(0, min(100, score))

    print("\n===== COMBINED VALIDATION RESULTS =====")
    for r in results:
        print(r)

    print(f"\nFINAL Confidence this ID is REAL: {score}/100")
    print(f"FINAL Confidence this ID is FAKE: {100-score}/100")
